In [4]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot


In [5]:
import subprocess

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr)

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Temurin-17.0.20.1+1 (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM Temurin-17.0.20.1+1 (build 17.0.20.1+1, mixed mode, sharing)



In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Healthcare Big Data Analytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print("Spark Version:", spark.version)

Spark Session Created Successfully!
Spark Version: 3.5.6


In [7]:
import os

dataset_path = r"E:\Healthcare_Big_Data_Project\dataset"

print("Files in dataset folder:")
print(os.listdir(dataset_path))

Files in dataset folder:
['admissions.csv', 'billing.csv', 'diagnoses.csv', 'hospitals.csv', 'patients.csv']


In [8]:
admissions_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\admissions.csv",
    header=True,
    inferSchema=True
)

admissions_df.show(5)

+--------------------+--------------------+----------+--------------+--------+----------+---------+--------------------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|        admission_id|          patient_id|admit_date|discharge_date|los_days|admit_type|ward_type|         hospital_id|discharge_type|num_procedures|charlson_index|hba1c|creatinine|haemoglobin|systolic_bp|readmitted_30d|readmitted_7d|
+--------------------+--------------------+----------+--------------+--------+----------+---------+--------------------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|bcb311d6-0808-498...|6960d399-8398-4db...|2024-01-23|    2024-01-27|       4|  Elective|  General|b22e7b51-2ea6-461...|     Recovered|             0|             0|  6.3|      0.86|       16.2|        157|             0|            0|
|996ac6db-a192-46a...|ec1999d4-f097-488...|2019-11-18|  

In [9]:
admissions_df.printSchema()

root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- admit_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- los_days: integer (nullable = true)
 |-- admit_type: string (nullable = true)
 |-- ward_type: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- discharge_type: string (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- charlson_index: integer (nullable = true)
 |-- hba1c: double (nullable = true)
 |-- creatinine: double (nullable = true)
 |-- haemoglobin: double (nullable = true)
 |-- systolic_bp: integer (nullable = true)
 |-- readmitted_30d: integer (nullable = true)
 |-- readmitted_7d: integer (nullable = true)



In [10]:
from pyspark.sql.functions import col, sum

admissions_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in admissions_df.columns
    ]
).show()

+------------+----------+----------+--------------+--------+----------+---------+-----------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|admission_id|patient_id|admit_date|discharge_date|los_days|admit_type|ward_type|hospital_id|discharge_type|num_procedures|charlson_index|hba1c|creatinine|haemoglobin|systolic_bp|readmitted_30d|readmitted_7d|
+------------+----------+----------+--------------+--------+----------+---------+-----------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|           0|         0|         0|             0|       0|         0|        0|          0|             0|             0|             0|    0|         0|          0|          0|             0|            0|
+------------+----------+----------+--------------+--------+----------+---------+-----------+--------------+--------------+--------------+-----+----------+---------

In [11]:
admissions_df.select("admit_type").distinct().show()

+----------+
|admit_type|
+----------+
|  Elective|
|       OPD|
| Emergency|
+----------+



In [12]:
admissions_df.select("ward_type").distinct().show()

+---------+
|ward_type|
+---------+
|      HDU|
|  General|
|      ICU|
|     NICU|
+---------+



In [13]:
admissions_df.select("discharge_type").distinct().show()

+--------------+
|discharge_type|
+--------------+
|          LAMA|
|       Expired|
|      Referred|
|     Recovered|
+--------------+



In [14]:
admissions_df.select("readmitted_30d").distinct().show()

+--------------+
|readmitted_30d|
+--------------+
|             1|
|             0|
+--------------+



In [15]:
admissions_df.select(
    "los_days",
    "num_procedures",
    "charlson_index",
    "hba1c",
    "creatinine",
    "haemoglobin",
    "systolic_bp"
).describe().show()

+-------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|summary|         los_days|    num_procedures|    charlson_index|             hba1c|        creatinine|       haemoglobin|      systolic_bp|
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|  count|           120000|            120000|            120000|            120000|            120000|            120000|           120000|
|   mean|6.852708333333333|1.5237416666666668|1.8948583333333333|5.8698616666666625| 1.070463166666668|11.840079166666678|       134.628075|
| stddev|5.708269005082808|1.5421973997202985|1.6103804439656517|1.0616619394049915|0.8961354581198278| 2.171929339176292|20.75654443973882|
|    min|                1|                 0|                 0|               4.0|               0.4|               4.0|               70|
|    max|    

In [16]:
cleaned_admissions_df = admissions_df.dropDuplicates()

print("Original records:", admissions_df.count())
print("Records after removing duplicates:", cleaned_admissions_df.count())

Original records: 120000
Records after removing duplicates: 120000


In [17]:
cleaned_admissions_df = admissions_df.dropDuplicates()

print("Cleaned admissions dataset:")
cleaned_admissions_df.show(5)

Cleaned admissions dataset:
+--------------------+--------------------+----------+--------------+--------+----------+---------+--------------------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|        admission_id|          patient_id|admit_date|discharge_date|los_days|admit_type|ward_type|         hospital_id|discharge_type|num_procedures|charlson_index|hba1c|creatinine|haemoglobin|systolic_bp|readmitted_30d|readmitted_7d|
+--------------------+--------------------+----------+--------------+--------+----------+---------+--------------------+--------------+--------------+--------------+-----+----------+-----------+-----------+--------------+-------------+
|77df74f0-2da3-458...|54406918-412d-4c3...|2021-09-21|    2021-10-02|      11| Emergency|      ICU|c522d48a-f9a9-41c...|      Referred|             2|             0|  5.7|      0.96|       10.8|        153|             0|            0|
|6949a031-ba46-474...|825aa5

In [18]:
patients_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\patients.csv",
    header=True,
    inferSchema=True
)

patients_df.show(5)

+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|          patient_id|age|gender|        state|bpl_card|insurance_type|comorbidity_count|prev_admissions|
+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|2b179f89-9af7-45c...| 36|     F|    Telangana|   false|       Private|                0|              0|
|ecaeb9f3-d263-471...| 56|     F|Uttar Pradesh|   false|           ESI|                0|              0|
|0160a5b6-877e-41e...| 67|     F|    Telangana|   false|           ESI|                3|              0|
|216875fa-b49d-4c4...| 53|     M|      Haryana|   false|      Ayushman|                3|              0|
|a2f3181d-e3f2-4fd...| 60|     M|  Maharashtra|    true|      Ayushman|                4|              1|
+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
only showing top 5 rows



In [19]:
patients_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in patients_df.columns
    ]
).show()

+----------+---+------+-----+--------+--------------+-----------------+---------------+
|patient_id|age|gender|state|bpl_card|insurance_type|comorbidity_count|prev_admissions|
+----------+---+------+-----+--------+--------------+-----------------+---------------+
|         0|  0|     0|    0|       0|             0|                0|              0|
+----------+---+------+-----+--------+--------------+-----------------+---------------+



In [20]:
print("Original patient records:", patients_df.count())
print("Records after removing duplicates:", patients_df.dropDuplicates().count())

Original patient records: 86400
Records after removing duplicates: 86400


In [21]:
cleaned_patients_df = patients_df.dropDuplicates()

print("Cleaned patients dataset:")
cleaned_patients_df.show(5)

Cleaned patients dataset:
+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|          patient_id|age|gender|        state|bpl_card|insurance_type|comorbidity_count|prev_admissions|
+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|fab45ba3-9100-486...| 46|     M|   Tamil Nadu|   false|      Ayushman|                0|              0|
|9bfaee54-0707-4f0...| 38|     M|    Karnataka|    true|      Ayushman|                1|              0|
|72cfa4db-8830-498...| 11|     F|   Tamil Nadu|    true|           ESI|                0|              0|
|bba64801-69f0-4c0...| 71|     F|    Telangana|   false|      Ayushman|                1|              0|
|186b650c-c86f-48c...| 56|     F|Uttar Pradesh|    true|      Ayushman|                2|              0|
+--------------------+---+------+-------------+--------+--------------+-----------------+---------------+
only showing top 5 r

In [22]:
hospitals_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\hospitals.csv",
    header=True,
    inferSchema=True
)

hospitals_df.show(5)

+--------------------+--------------------+-------------+-----+----+--------+
|         hospital_id|                name|        state| tier|beds|teaching|
+--------------------+--------------------+-------------+-----+----+--------+
|df0f948a-2202-459...|Government Genera...|  Maharashtra|tier2| 314|    true|
|3d59f4a6-ce9c-494...|Government Genera...|Uttar Pradesh|tier2| 481|    true|
|aac12881-4dea-42f...|Government Genera...|   Tamil Nadu|tier2| 428|    true|
|28879338-cd09-48b...|Government Genera...|    Karnataka|tier2| 304|    true|
|2f5c9dbb-cef7-46a...|Government Genera...|  West Bengal|tier2| 632|    true|
+--------------------+--------------------+-------------+-----+----+--------+
only showing top 5 rows



In [23]:
hospitals_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in hospitals_df.columns
    ]
).show()

+-----------+----+-----+----+----+--------+
|hospital_id|name|state|tier|beds|teaching|
+-----------+----+-----+----+----+--------+
|          0|   0|    0|   0|   0|       0|
+-----------+----+-----+----+----+--------+



In [24]:
print("Original hospital records:", hospitals_df.count())
print("Records after removing duplicates:", hospitals_df.dropDuplicates().count())

Original hospital records: 33
Records after removing duplicates: 33


In [25]:
cleaned_hospitals_df = hospitals_df.dropDuplicates()

print("Cleaned hospitals dataset:")
cleaned_hospitals_df.show(5)

Cleaned hospitals dataset:
+--------------------+--------------------+-----------+-----+----+--------+
|         hospital_id|                name|      state| tier|beds|teaching|
+--------------------+--------------------+-----------+-----+----+--------+
|07be5257-4d9e-4a9...|Government Genera...|    Gujarat|tier2| 230|    true|
|ad0ba923-bc16-42a...|Apollo/Manipal (P...|  Karnataka|tier1| 999|    true|
|df0f948a-2202-459...|Government Genera...|Maharashtra|tier2| 314|    true|
|e698154c-39f9-44e...|Government Genera...|     Punjab|tier2| 548|   false|
|f0bb4c45-e80c-42f...|Government Genera...|     Kerala|tier2| 774|    true|
+--------------------+--------------------+-----------+-----+----+--------+
only showing top 5 rows



In [26]:
diagnoses_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\diagnoses.csv",
    header=True,
    inferSchema=True
)

diagnoses_df.show(5)

+--------------------+--------------------+----------+--------------------+---------+--------------+
|             diag_id|        admission_id|icd10_code|           diag_desc|diag_rank| diag_category|
+--------------------+--------------------+----------+--------------------+---------+--------------+
|3b448b98-6b78-4c7...|bcb311d6-0808-498...|       E87|Metabolic / elect...|        1|     Endocrine|
|137c0404-f1b6-4ac...|996ac6db-a192-46a...|       J45|              Asthma|        1|   Respiratory|
|4bfb76a1-12ed-4cb...|cc4849fa-b740-487...|       G40|            Epilepsy|        1|  Neurological|
|754ec3d2-e940-434...|cc4849fa-b740-487...|       A15|Pulmonary tubercu...|        2|    Infectious|
|f4fa0d7b-0a68-4dd...|cc4849fa-b740-487...|       I48| Atrial fibrillation|        3|Cardiovascular|
+--------------------+--------------------+----------+--------------------+---------+--------------+
only showing top 5 rows



In [27]:
diagnoses_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in diagnoses_df.columns
    ]
).show()

+-------+------------+----------+---------+---------+-------------+
|diag_id|admission_id|icd10_code|diag_desc|diag_rank|diag_category|
+-------+------------+----------+---------+---------+-------------+
|      0|           0|         0|        0|        0|            0|
+-------+------------+----------+---------+---------+-------------+



In [28]:
print("Original diagnosis records:", diagnoses_df.count())
print("Records after removing duplicates:", diagnoses_df.dropDuplicates().count())

Original diagnosis records: 271341
Records after removing duplicates: 271341


In [29]:
cleaned_diagnoses_df = diagnoses_df.dropDuplicates()

print("Cleaned diagnoses dataset:")
cleaned_diagnoses_df.show(5)

Cleaned diagnoses dataset:
+--------------------+--------------------+----------+--------------------+---------+----------------+
|             diag_id|        admission_id|icd10_code|           diag_desc|diag_rank|   diag_category|
+--------------------+--------------------+----------+--------------------+---------+----------------+
|f9cd8785-53e7-4c4...|57bf7fd2-bb5d-485...|       J44|                COPD|        4|     Respiratory|
|38611c37-ab74-4a9...|ba82c8d4-181e-4de...|       E11|Type 2 diabetes m...|        1|       Endocrine|
|b5288a34-ff5d-454...|068b5463-134b-4d5...|       K80|      Cholelithiasis|        3|Gastrointestinal|
|5d2f32a9-a44b-4c8...|422e67fe-6e54-4e7...|       E10|Type 1 diabetes m...|        2|       Endocrine|
|0175cc4c-ee64-43e...|95e978f6-bc0e-4f4...|       K57|Diverticular dise...|        4|Gastrointestinal|
+--------------------+--------------------+----------+--------------------+---------+----------------+
only showing top 5 rows



In [30]:
billing_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\billing.csv",
    header=True,
    inferSchema=True
)

billing_df.show(5)

+--------------------+--------------------+--------------+----------------+-----------------+-------------+
|             bill_id|        admission_id|total_cost_inr|govt_subsidy_inr|out_of_pocket_inr|cost_category|
+--------------------+--------------------+--------------+----------------+-----------------+-------------+
|68e19242-4126-4a7...|bcb311d6-0808-498...|         18492|           16630|             1862|          Lab|
|b740202e-1d7a-45b...|996ac6db-a192-46a...|          1001|               0|             1001|          Lab|
|f60fecaa-e769-42e...|cc4849fa-b740-487...|         59696|           48650|            11046|         Room|
|32f41176-c088-4e9...|b38cdd03-165c-428...|         72588|               0|            72588|          Lab|
|cf80fba9-7fec-4b2...|6a154ec0-92cc-43f...|          9510|               0|             9510|          Lab|
+--------------------+--------------------+--------------+----------------+-----------------+-------------+
only showing top 5 rows



In [31]:
billing_df.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in billing_df.columns
    ]
).show()

+-------+------------+--------------+----------------+-----------------+-------------+
|bill_id|admission_id|total_cost_inr|govt_subsidy_inr|out_of_pocket_inr|cost_category|
+-------+------------+--------------+----------------+-----------------+-------------+
|      0|           0|             0|               0|                0|            0|
+-------+------------+--------------+----------------+-----------------+-------------+



In [32]:
print("Original billing records:", billing_df.count())
print("Records after removing duplicates:", billing_df.dropDuplicates().count())

Original billing records: 120000
Records after removing duplicates: 120000


In [33]:
cleaned_billing_df = billing_df.dropDuplicates()

print("Cleaned billing dataset:")
cleaned_billing_df.show(5)

Cleaned billing dataset:
+--------------------+--------------------+--------------+----------------+-----------------+-------------+
|             bill_id|        admission_id|total_cost_inr|govt_subsidy_inr|out_of_pocket_inr|cost_category|
+--------------------+--------------------+--------------+----------------+-----------------+-------------+
|c9cd63a7-a0d5-4d6...|7959dd8c-b1d8-467...|         14956|               0|            14956|         Room|
|79f27344-be11-4cb...|7db0baea-a405-4b0...|          7181|               0|             7181|          Lab|
|5ca3803d-25c8-47a...|11ab7c15-2d2f-427...|         68589|           42740|            25849|    Procedure|
|95a6ea26-a943-4da...|aa558cdb-ab57-49c...|         10383|            7380|             3003|    Procedure|
|a56367ec-48b5-4cf...|b2d40824-5a52-4ab...|        336761|               0|           336761|     Pharmacy|
+--------------------+--------------------+--------------+----------------+-----------------+-------------+
onl

In [34]:
disease_analysis = cleaned_diagnoses_df.groupBy(
    "diag_category"
).count().orderBy(
    "count", ascending=False
)

disease_analysis.show()

+----------------+-----+
|   diag_category|count|
+----------------+-----+
|  Cardiovascular|53410|
|       Endocrine|35491|
|     Respiratory|30648|
|      Infectious|30397|
|Gastrointestinal|23601|
|        Neoplasm|21776|
|          Injury|17278|
|    Neurological|17111|
|   Genitourinary|16591|
|       Obstetric|12973|
|       Perinatal|12065|
+----------------+-----+



In [35]:
from pyspark.sql.functions import countDistinct

In [36]:
patients_by_disease = cleaned_diagnoses_df.join(
    cleaned_admissions_df,
    on="admission_id",
    how="inner"
).groupBy(
    "diag_category"
).agg(
    countDistinct("patient_id").alias("unique_patients")
).orderBy(
    "unique_patients", ascending=False
)

patients_by_disease.show()

+----------------+---------------+
|   diag_category|unique_patients|
+----------------+---------------+
|  Cardiovascular|          33136|
|       Endocrine|          26097|
|      Infectious|          23534|
|     Respiratory|          22351|
|Gastrointestinal|          19229|
|        Neoplasm|          17322|
|          Injury|          14870|
|    Neurological|          14621|
|   Genitourinary|          14425|
|       Obstetric|          11674|
|       Perinatal|          10888|
+----------------+---------------+



In [37]:
age_stay_analysis = cleaned_admissions_df.join(
    cleaned_patients_df,
    on="patient_id",
    how="inner"
).select(
    "age",
    "los_days"
).agg(
    {"age": "avg", "los_days": "avg"}
)

age_stay_analysis.show()

+-----------------+-----------------+
|         avg(age)|    avg(los_days)|
+-----------------+-----------------+
|48.04986666666667|6.852708333333333|
+-----------------+-----------------+



In [38]:
readmission_analysis = cleaned_admissions_df.groupBy(
    "readmitted_30d"
).count().orderBy(
    "readmitted_30d"
)

readmission_analysis.show()

+--------------+------+
|readmitted_30d| count|
+--------------+------+
|             0|105790|
|             1| 14210|
+--------------+------+



In [39]:
total_admissions = cleaned_admissions_df.count()

readmitted_30d_count = cleaned_admissions_df.filter(
    col("readmitted_30d") == 1
).count()

readmission_rate = (readmitted_30d_count / total_admissions) * 100

print("Total admissions:", total_admissions)
print("30-day readmissions:", readmitted_30d_count)
print("30-day readmission rate:", round(readmission_rate, 2), "%")

Total admissions: 120000
30-day readmissions: 14210
30-day readmission rate: 11.84 %


In [40]:
from pyspark.sql.functions import avg

readmission_by_ward = cleaned_admissions_df.groupBy(
    "ward_type"
).agg(
    avg("readmitted_30d").alias("readmission_rate")
).orderBy(
    "readmission_rate", ascending=False
)

readmission_by_ward.show()

+---------+-------------------+
|ward_type|   readmission_rate|
+---------+-------------------+
|      ICU|0.23760380166082656|
|     NICU| 0.1660505237215034|
|      HDU|0.13308606273967435|
|  General|  0.079004272613589|
+---------+-------------------+



In [41]:
readmission_by_admit_type = cleaned_admissions_df.groupBy(
    "admit_type"
).agg(
    avg("readmitted_30d").alias("readmission_rate")
).orderBy(
    "readmission_rate", ascending=False
)

readmission_by_admit_type.show()

+----------+-------------------+
|admit_type|   readmission_rate|
+----------+-------------------+
| Emergency|0.14816651638112413|
|  Elective| 0.0825297862784984|
|       OPD|0.07861856859101685|
+----------+-------------------+



In [42]:
readmission_by_disease = cleaned_diagnoses_df.join(
    cleaned_admissions_df,
    on="admission_id",
    how="inner"
).groupBy(
    "diag_category"
).agg(
    avg("readmitted_30d").alias("readmission_rate")
).orderBy(
    "readmission_rate", ascending=False
)

readmission_by_disease.show()

+----------------+-------------------+
|   diag_category|   readmission_rate|
+----------------+-------------------+
|     Respiratory| 0.1774667188723571|
|       Perinatal|0.17397430584334853|
|  Cardiovascular|0.17032390938026587|
|        Neoplasm|0.16251836884643645|
|   Genitourinary|0.15466216623470555|
|          Injury|0.14608172242157658|
|       Endocrine|0.13933109802485136|
|      Infectious|  0.139191367569168|
|    Neurological|0.12115013733855415|
|Gastrointestinal| 0.1205033685013347|
|       Obstetric|0.11053726971402143|
+----------------+-------------------+



In [43]:
readmission_7d_analysis = cleaned_admissions_df.groupBy(
    "readmitted_7d"
).count().orderBy(
    "readmitted_7d"
)

readmission_7d_analysis.show()

+-------------+------+
|readmitted_7d| count|
+-------------+------+
|            0|118994|
|            1|  1006|
+-------------+------+



In [44]:
total_admissions = cleaned_admissions_df.count()

readmitted_7d_count = cleaned_admissions_df.filter(
    col("readmitted_7d") == 1
).count()

readmission_7d_rate = (readmitted_7d_count / total_admissions) * 100

print("Total admissions:", total_admissions)
print("7-day readmissions:", readmitted_7d_count)
print("7-day readmission rate:", round(readmission_7d_rate, 2), "%")

Total admissions: 120000
7-day readmissions: 1006
7-day readmission rate: 0.84 %


In [45]:
average_stay_by_ward = cleaned_admissions_df.groupBy(
    "ward_type"
).agg(
    avg("los_days").alias("average_stay")
).orderBy(
    "average_stay", ascending=False
)

average_stay_by_ward.show()

+---------+------------------+
|ward_type|      average_stay|
+---------+------------------+
|     NICU|16.612138016019717|
|      ICU|12.914126626026016|
|      HDU| 6.258565411454077|
|  General| 4.505575239683202|
+---------+------------------+



In [46]:
cleaned_admissions_df.createOrReplaceTempView("admissions")

In [47]:
spark.sql("""
    SELECT ward_type,
           COUNT(*) AS total_admissions
    FROM admissions
    GROUP BY ward_type
    ORDER BY total_admissions DESC
""").show()

+---------+----------------+
|ward_type|total_admissions|
+---------+----------------+
|  General|           76768|
|      ICU|           20833|
|      HDU|           15907|
|     NICU|            6492|
+---------+----------------+



In [48]:
spark.sql("""
    SELECT admit_type,
           COUNT(*) AS total_admissions,
           SUM(readmitted_30d) AS readmitted_30d,
           ROUND(AVG(readmitted_30d) * 100, 2) AS readmission_rate
    FROM admissions
    GROUP BY admit_type
    ORDER BY readmission_rate DESC
""").show()

+----------+----------------+--------------+----------------+
|admit_type|total_admissions|readmitted_30d|readmission_rate|
+----------+----------------+--------------+----------------+
| Emergency|           66540|          9859|           14.82|
|  Elective|           37853|          3124|            8.25|
|       OPD|           15607|          1227|            7.86|
+----------+----------------+--------------+----------------+



In [49]:
hospitals_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\hospitals.csv",
    header=True,
    inferSchema=True
)

cleaned_hospitals_df = hospitals_df.dropDuplicates()

print("Total hospitals:", cleaned_hospitals_df.count())

Total hospitals: 33


In [50]:
hospital_admissions = cleaned_admissions_df.join(
    cleaned_hospitals_df,
    on="hospital_id",
    how="inner"
).groupBy(
    "name",
    "state",
    "tier"
).count().orderBy(
    "count", ascending=False
)

hospital_admissions.show(10, truncate=False)

+---------------------------------------+--------------+-----+-----+
|name                                   |state         |tier |count|
+---------------------------------------+--------------+-----+-----+
|Apollo/Manipal (Private) Andhra Pradesh|Andhra Pradesh|tier1|14497|
|District Hospital Bihar                |Bihar         |tier3|7439 |
|District Hospital Uttar Pradesh        |Uttar Pradesh |tier3|7369 |
|Apollo/Manipal (Private) Karnataka     |Karnataka     |tier1|7245 |
|District Hospital Odisha               |Odisha        |tier3|7125 |
|Apollo/Manipal (Private) Telangana     |Telangana     |tier1|3720 |
|Government General Hospital Telangana  |Telangana     |tier2|3701 |
|Government General Hospital Maharashtra|Maharashtra   |tier2|3692 |
|Government General Hospital Bihar      |Bihar         |tier2|3665 |
|Government General Hospital Odisha     |Odisha        |tier2|3663 |
+---------------------------------------+--------------+-----+-----+
only showing top 10 rows



In [51]:
admissions_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\admissions.csv",
    header=True,
    inferSchema=True
)

cleaned_admissions_df = admissions_df.dropDuplicates()

print("Total admissions:", cleaned_admissions_df.count())

Total admissions: 120000


In [52]:
billing_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\billing.csv",
    header=True,
    inferSchema=True
)

cleaned_billing_df = billing_df.dropDuplicates()

print("Total billing records:", cleaned_billing_df.count())

Total billing records: 120000


In [53]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Healthcare Big Data Analytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started successfully!")
print(spark.version)

Spark started successfully!
3.5.6


In [54]:
print("Total billing records:", cleaned_billing_df.count())

Total billing records: 120000


In [55]:
from pyspark.sql.functions import avg, sum

billing_summary = cleaned_billing_df.agg(
    avg("total_cost_inr").alias("average_total_cost"),
    avg("govt_subsidy_inr").alias("average_government_subsidy"),
    avg("out_of_pocket_inr").alias("average_out_of_pocket")
)

billing_summary.show(truncate=False)

+------------------+--------------------------+---------------------+
|average_total_cost|average_government_subsidy|average_out_of_pocket|
+------------------+--------------------------+---------------------+
|95779.53165       |47606.16778333333         |48173.36386666667    |
+------------------+--------------------------+---------------------+



In [56]:
from pyspark.sql.functions import avg

billing_hospital_df = cleaned_billing_df.join(
    cleaned_admissions_df,
    on="admission_id",
    how="inner"
).join(
    cleaned_hospitals_df,
    on="hospital_id",
    how="inner"
)

print("Total joined records:", billing_hospital_df.count())

billing_hospital_df.select(
    "name", "tier", "total_cost_inr",
    "govt_subsidy_inr", "out_of_pocket_inr"
).show(5, truncate=False)

Total joined records: 120000
+------------------------------------------+-----+--------------+----------------+-----------------+
|name                                      |tier |total_cost_inr|govt_subsidy_inr|out_of_pocket_inr|
+------------------------------------------+-----+--------------+----------------+-----------------+
|District Hospital Bihar                   |tier3|14956         |0               |14956            |
|Government General Hospital Madhya Pradesh|tier2|7181          |0               |7181             |
|Government General Hospital Andhra Pradesh|tier2|68589         |42740           |25849            |
|District Hospital Odisha                  |tier3|10383         |7380            |3003             |
|Government General Hospital Tamil Nadu    |tier2|336761        |0               |336761           |
+------------------------------------------+-----+--------------+----------------+-----------------+
only showing top 5 rows



In [57]:
from pyspark.sql.functions import avg, round

tier_billing_summary = billing_hospital_df.groupBy("tier").agg(
    round(avg("total_cost_inr"), 2).alias("average_total_cost"),
    round(avg("govt_subsidy_inr"), 2).alias("average_government_subsidy"),
    round(avg("out_of_pocket_inr"), 2).alias("average_patient_expense")
).orderBy("tier")

tier_billing_summary.show(truncate=False)

+-----+------------------+--------------------------+-----------------------+
|tier |average_total_cost|average_government_subsidy|average_patient_expense|
+-----+------------------+--------------------------+-----------------------+
|tier1|220492.83         |100508.19                 |119984.64              |
|tier2|74723.57          |40426.41                  |34297.15               |
|tier3|27983.18          |16229.23                  |11753.95               |
+-----+------------------+--------------------------+-----------------------+



In [58]:
from pyspark.sql.functions import avg, round

state_billing_summary = billing_hospital_df.groupBy("state").agg(
    round(avg("total_cost_inr"), 2).alias("average_total_cost"),
    round(avg("govt_subsidy_inr"), 2).alias("average_subsidy"),
    round(avg("out_of_pocket_inr"), 2).alias("average_patient_expense")
).orderBy("average_total_cost", ascending=False)

state_billing_summary.show(10, truncate=False)

+--------------+------------------+---------------+-----------------------+
|state         |average_total_cost|average_subsidy|average_patient_expense|
+--------------+------------------+---------------+-----------------------+
|Andhra Pradesh|190860.08         |87833.04       |103027.04              |
|Karnataka     |174846.17         |81473.19       |93372.97               |
|Telangana     |145308.76         |70445.85       |74862.91               |
|Uttar Pradesh |85966.65          |42778.79       |43187.86               |
|Tamil Nadu    |77530.83          |41115.9        |36414.93               |
|Punjab        |76410.35          |40694.56       |35715.79               |
|Haryana       |76259.39          |41788.69       |34470.7                |
|West Bengal   |75919.89          |41754.52       |34165.37               |
|Maharashtra   |74189.21          |42039.37       |32149.85               |
|Kerala        |52441.0           |28566.23       |23874.77               |
+-----------

In [59]:
from pyspark.sql.functions import col, when, avg, round

patient_admissions_df = cleaned_admissions_df.join(
    cleaned_patients_df.select("patient_id", "age"),
    on="patient_id",
    how="inner"
)

patient_age_df = patient_admissions_df.withColumn(
    "age_group",
    when(col("age") < 18, "Below 18")
    .when(col("age") < 40, "18–39")
    .when(col("age") < 60, "40–59")
    .otherwise("60 and above")
)

age_readmission_summary = patient_age_df.groupBy("age_group").agg(
    round(avg("readmitted_30d") * 100, 2)
    .alias("readmission_rate_percent"),
    col("age_group")
).orderBy("age_group")

age_readmission_summary.show(truncate=False)

+------------+------------------------+------------+
|age_group   |readmission_rate_percent|age_group   |
+------------+------------------------+------------+
|18–39       |5.78                    |18–39       |
|40–59       |10.03                   |40–59       |
|60 and above|19.59                   |60 and above|
|Below 18    |6.16                    |Below 18    |
+------------+------------------------+------------+



In [60]:

from pyspark.sql.functions import col, when, avg, round

patient_admissions_df = cleaned_admissions_df.join(
    cleaned_patients_df.select("patient_id", "age"),
    on="patient_id",
    how="inner"
)

patient_age_df = patient_admissions_df.withColumn(
    "age_group",
    when(col("age") < 18, "Below 18")
    .when(col("age") < 40, "18-39")
    .when(col("age") < 60, "40-59")
    .otherwise("60 and above")
)

age_readmission_summary = patient_age_df.groupBy("age_group").agg(
    round(avg("readmitted_30d") * 100, 2)
    .alias("readmission_rate_percent")
).orderBy("age_group")

age_readmission_summary.show(truncate=False)

+------------+------------------------+
|age_group   |readmission_rate_percent|
+------------+------------------------+
|18-39       |5.78                    |
|40-59       |10.03                   |
|60 and above|19.59                   |
|Below 18    |6.16                    |
+------------+------------------------+



In [61]:
patients_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\patients.csv",
    header=True,
    inferSchema=True
)

cleaned_patients_df = patients_df.dropDuplicates()

print("Total patient records:", cleaned_patients_df.count())

cleaned_patients_df.show(5, truncate=False)

Total patient records: 86400
+------------------------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|patient_id                          |age|gender|state        |bpl_card|insurance_type|comorbidity_count|prev_admissions|
+------------------------------------+---+------+-------------+--------+--------------+-----------------+---------------+
|fab45ba3-9100-4861-8d09-f4d675daefb1|46 |M     |Tamil Nadu   |false   |Ayushman      |0                |0              |
|9bfaee54-0707-4f07-880b-b6e2c60fa5df|38 |M     |Karnataka    |true    |Ayushman      |1                |0              |
|72cfa4db-8830-4988-b1f2-89138ebb0298|11 |F     |Tamil Nadu   |true    |ESI           |0                |0              |
|bba64801-69f0-4c0a-aebd-f8df32f7d988|71 |F     |Telangana    |false   |Ayushman      |1                |0              |
|186b650c-c86f-48c0-998c-1a5cdef3060b|56 |F     |Uttar Pradesh|true    |Ayushman      |2                |0           

In [62]:
gender_readmission_summary = patient_admissions_df.groupBy("gender").agg(
    round(avg("readmitted_30d") * 100, 2)
    .alias("readmission_rate_percent"),
    count("*").alias("total_admissions")
).orderBy("readmission_rate_percent", ascending=False)

gender_readmission_summary.show(truncate=False)

NameError: name 'count' is not defined

In [ ]:
from pyspark.sql.functions import count

In [ ]:
patient_admissions_df = cleaned_admissions_df.join(
    cleaned_patients_df.select("patient_id", "age", "gender"),
    on="patient_id",
    how="inner"
)

print("Total joined records:", patient_admissions_df.count())
patient_admissions_df.select("patient_id", "age", "gender").show(5)

In [ ]:
from pyspark.sql.functions import avg, round, count

gender_readmission_summary = patient_admissions_df.groupBy("gender").agg(
    round(avg("readmitted_30d") * 100, 2)
    .alias("readmission_rate_percent"),
    count("*").alias("total_admissions")
).orderBy("readmission_rate_percent", ascending=False)

gender_readmission_summary.show(truncate=False)

In [ ]:
previous_admission_summary = cleaned_patients_df.join(
    cleaned_admissions_df,
    on="patient_id",
    how="inner"
).groupBy("prev_admissions").agg(
    round(avg("readmitted_30d") * 100, 2)
    .alias("readmission_rate_percent"),
    count("*").alias("total_admissions")
).orderBy("prev_admissions")

previous_admission_summary.show(truncate=False)

In [ ]:
print("Total diagnosis records:", cleaned_diagnoses_df.count())
cleaned_diagnoses_df.show(5, truncate=False)

In [ ]:
diagnoses_df = spark.read.csv(
    r"E:\Healthcare_Big_Data_Project\dataset\diagnoses.csv",
    header=True,
    inferSchema=True
)

cleaned_diagnoses_df = diagnoses_df.dropDuplicates()

print("Total diagnosis records:", cleaned_diagnoses_df.count())

cleaned_diagnoses_df.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import count

disease_summary = cleaned_diagnoses_df.groupBy(
    "diag_category"
).agg(
    count("*").alias("total_diagnoses")
).orderBy(
    "total_diagnoses", ascending=False
)

disease_summary.show(truncate=False)

In [ ]:

# Check patient and admission columns

print("PATIENTS DATASET")
cleaned_patients_df.printSchema()
cleaned_patients_df.show(5, truncate=False)

print("ADMISSIONS DATASET")
cleaned_admissions_df.printSchema()
cleaned_admissions_df.show(5, truncate=False)


In [ ]:

from pyspark.sql.functions import avg, min, max, round, count

# 1. Analyse patient age
age_summary = cleaned_patients_df.agg(
    count("patient_id").alias("total_patients"),
    round(avg("age"), 2).alias("average_age"),
    min("age").alias("minimum_age"),
    max("age").alias("maximum_age")
)

print("PATIENT AGE SUMMARY")
age_summary.show(truncate=False)


# 2. Analyse hospital stay duration
stay_summary = cleaned_admissions_df.agg(
    count("admission_id").alias("total_admissions"),
    round(avg("los_days"), 2).alias("average_stay_days"),
    min("los_days").alias("minimum_stay_days"),
    max("los_days").alias("maximum_stay_days")
)

print("HOSPITAL STAY SUMMARY")
stay_summary.show(truncate=False)


In [ ]:

from pyspark.sql.functions import sum as spark_sum

readmission_summary = cleaned_admissions_df.agg(
    count("admission_id").alias("total_admissions"),

    spark_sum("readmitted_7d").alias("readmissions_7d"),

    spark_sum("readmitted_30d").alias("readmissions_30d")
)

print("READMISSION SUMMARY")
readmission_summary.show(truncate=False)


In [ ]:

from pyspark.sql.functions import count, sum as spark_sum, round

readmission_by_type = cleaned_admissions_df.groupBy(
    "admit_type"
).agg(
    count("admission_id").alias("total_admissions"),
    spark_sum("readmitted_7d").alias("readmissions_7d"),
    spark_sum("readmitted_30d").alias("readmissions_30d")
).withColumn(
    "readmission_rate_30d_pct",
    round(
        spark_sum("readmitted_30d") / count("admission_id") * 100,
        2
    )
).orderBy(
    "readmission_rate_30d_pct",
    ascending=False
)

readmission_by_type.show(truncate=False)


In [ ]:

from pyspark.sql.functions import count, sum as spark_sum, round, col

# Analyse readmissions by admission type
readmission_by_type = cleaned_admissions_df.groupBy(
    "admit_type"
).agg(
    count("admission_id").alias("total_admissions"),
    spark_sum("readmitted_7d").alias("readmissions_7d"),
    spark_sum("readmitted_30d").alias("readmissions_30d")
).withColumn(
    "readmission_rate_30d_pct",
    round(
        col("readmissions_30d") / col("total_admissions") * 100,
        2
    )
).orderBy(
    "readmission_rate_30d_pct",
    ascending=False
)

# Display the results
readmission_by_type.show(truncate=False)


In [ ]:

# Register DataFrames as temporary SQL views

cleaned_admissions_df.createOrReplaceTempView("admissions")
cleaned_patients_df.createOrReplaceTempView("patients")
cleaned_diagnoses_df.createOrReplaceTempView("diagnoses")

print("Temporary SQL views created successfully!")


In [ ]:

admissions_by_type = spark.sql("""
    SELECT
        admit_type,
        COUNT(*) AS total_admissions
    FROM admissions
    GROUP BY admit_type
    ORDER BY total_admissions DESC
""")

admissions_by_type.show(truncate=False)


In [ ]:

average_stay_by_ward = spark.sql("""
    SELECT
        ward_type,
        COUNT(*) AS total_admissions,
        ROUND(AVG(los_days), 2) AS average_stay_days
    FROM admissions
    GROUP BY ward_type
    ORDER BY average_stay_days DESC
""")

average_stay_by_ward.show(truncate=False)


In [ ]:

# Spark SQL: Count diagnoses by disease category

disease_sql_summary = spark.sql("""
    SELECT
        diag_category,
        COUNT(*) AS total_diagnoses
    FROM diagnoses
    GROUP BY diag_category
    ORDER BY total_diagnoses DESC
""")

disease_sql_summary.show(truncate=False)


In [ ]:

from pyspark.sql.functions import col

# Transformation 1: Filter emergency admissions
emergency_admissions = cleaned_admissions_df.filter(
    col("admit_type") == "Emergency"
)

# Transformation 2: Select relevant columns
emergency_summary_df = emergency_admissions.select(
    "admission_id",
    "patient_id",
    "ward_type",
    "los_days",
    "readmitted_30d"
)

# Action 1: Count emergency admissions
print("Total emergency admissions:", emergency_admissions.count())

# Action 2: Display sample records
emergency_summary_df.show(5, truncate=False)


In [ ]:

from pyspark.sql.functions import count, sum as spark_sum, round, col

readmission_by_ward = cleaned_admissions_df.groupBy(
    "ward_type"
).agg(
    count("admission_id").alias("total_admissions"),
    spark_sum("readmitted_30d").alias("readmissions_30d")
).withColumn(
    "readmission_rate_pct",
    round(
        col("readmissions_30d") / col("total_admissions") * 100,
        2
    )
).orderBy(
    "readmission_rate_pct",
    ascending=False
)

readmission_by_ward.show(truncate=False)


In [63]:
print(spark)